In [ ]:
# ═══════════════════════════════════════════════════════════════
# CELL 1: Imports
# ═══════════════════════════════════════════════════════════════
import sys, os
sys.path.insert(0, os.path.abspath(os.path.join(os.path.dirname('__file__'), '..', 'scripts')))

import pandas as pd
import numpy as np
import warnings
warnings.filterwarnings('ignore')

from eda import *

print('All imports successful. eda.py loaded.')

## yfinance Data Schema

| Exchange | Suffix | Example |
|---|---|---|
| NSE (India) | `.NS` | `RELIANCE.NS`, `TCS.NS` |
| BSE (India) | `.BO` | `RELIANCE.BO` |
| US | *(none)* | `AAPL`, `MSFT` |
| Indices | `^` prefix | `^NSEI`, `^GSPC` |
| Crypto | `-USD` | `BTC-USD` |

- Columns: `Open, High, Low, Close, Volume` (DatetimeIndex)
- Schema is **identical** across all exchanges — only ticker format differs
- `auto_adjust=True` → adjusted Close (splits & dividends baked in)

In [ ]:
# ═══════════════════════════════════════════════════════════════
# CELL 2: Schema Demo
# ═══════════════════════════════════════════════════════════════
df_demo = fetch_data('RELIANCE.NS', period='5d', interval='1d')
print(f'Columns : {list(df_demo.columns)}')
print(f'Index   : {type(df_demo.index).__name__}')
print(f'Dtypes  :\n{df_demo.dtypes}')
print(f'\nFirst 3 rows:\n{df_demo.head(3)}')

In [ ]:
# ═══════════════════════════════════════════════════════════════
# CELL 3: Fetch NSE Data (2 years)
# ═══════════════════════════════════════════════════════════════
TICKER   = 'RELIANCE.NS'
EXCHANGE = 'NSE'

sym = normalize_ticker(TICKER, EXCHANGE)
print(f'Normalized ticker: {sym}')

df = fetch_data(sym, period='2y', interval='1d')
print(f'Fetched {len(df)} rows  |  {df.index[0].date()} → {df.index[-1].date()}')
print(f'Columns: {list(df.columns)}')
df.head()

In [ ]:
# ═══════════════════════════════════════════════════════════════
# CELL 4: Add All Technical Indicators
# ═══════════════════════════════════════════════════════════════
df = add_all_technicals(df)

tech_cols = [c for c in df.columns if c not in ['Open','High','Low','Close','Volume']]
print('Added indicators:', tech_cols)
df[['Close','Daily_Return','SMA_20','RSI','MACD','ATR','Vol_20d']].tail(3)

In [ ]:
# ═══════════════════════════════════════════════════════════════
# CELL 5: Summary Statistics
# ═══════════════════════════════════════════════════════════════
price_stats = compute_summary_stats(df['Close'])
ret_stats   = compute_summary_stats(df['Daily_Return'])

print('=== PRICE STATS ===')
for k, v in price_stats.items():
    print(f'  {k:25s}: {v:.4f}' if isinstance(v, float) else f'  {k:25s}: {v}')

print('\n=== RETURN STATS ===')
for k, v in ret_stats.items():
    print(f'  {k:25s}: {v:.6f}' if isinstance(v, float) else f'  {k:25s}: {v}')

In [ ]:
# ═══════════════════════════════════════════════════════════════
# CELL 6: Stationarity Tests (ADF & KPSS)
# ═══════════════════════════════════════════════════════════════
for label, series in [
    ('Raw Close',          df['Close']),
    ('Daily Returns',      df['Daily_Return']),
    ('Log Returns',        df['Log_Return']),
    ('First Diff (Close)', df['Close'].diff().dropna()),
]:
    r = run_stationarity_tests(series)
    print(f'\n--- {label} ---')
    print(f'  ADF : {r["adf_statistic"]:7.4f}  p={r["adf_pvalue"]:.4f}  → {r["adf_conclusion"]}')
    print(f'  KPSS: {r["kpss_statistic"]:7.4f}  p={r["kpss_pvalue"]:.4f}  → {r["kpss_conclusion"]}')

In [ ]:
# ═══════════════════════════════════════════════════════════════
# CELL 7: Time-Series Decomposition
# ═══════════════════════════════════════════════════════════════
decomp = decompose_series(df['Close'], model='additive', period=252)
print(f'Trend  : {decomp.trend.dropna().iloc[0]:.2f} → {decomp.trend.dropna().iloc[-1]:.2f}')
print(f'Seasonal range: [{decomp.seasonal.min():.2f}, {decomp.seasonal.max():.2f}]')
print(f'Residual std  : {decomp.resid.dropna().std():.4f}')

fig_decomp = plot_decomposition(decomp, ticker=sym)
fig_decomp.show()

In [ ]:
# ═══════════════════════════════════════════════════════════════
# CELL 8: ARIMA Model
# ═══════════════════════════════════════════════════════════════
arima_fit, arima_forecast = fit_arima(df['Close'], order=(1, 1, 1), forecast_steps=30)

print(f'ARIMA(1,1,1)  AIC={arima_fit.aic:.2f}  BIC={arima_fit.bic:.2f}')
print(arima_fit.summary().tables[1])
print('\nForecast (next 5 days):')
print(arima_forecast.head())

plot_forecast(df['Close'], arima_forecast, ticker=f'{sym} ARIMA(1,1,1)').show()

In [ ]:
# ═══════════════════════════════════════════════════════════════
# CELL 9: SARIMA Model
# ═══════════════════════════════════════════════════════════════
sarima_fit, sarima_forecast = fit_sarima(
    df['Close'], order=(1, 1, 1), seasonal_order=(1, 1, 1, 252), forecast_steps=30
)

print(f'SARIMA(1,1,1)(1,1,1,252)  AIC={sarima_fit.aic:.2f}  BIC={sarima_fit.bic:.2f}')
print('\nForecast (next 5 days):')
print(sarima_forecast.head())

plot_forecast(df['Close'], sarima_forecast, ticker=f'{sym} SARIMA').show()

In [ ]:
# ═══════════════════════════════════════════════════════════════
# CELL 10: Residual Diagnostics
# ═══════════════════════════════════════════════════════════════
for label, model in [('ARIMA', arima_fit), ('SARIMA', sarima_fit)]:
    diag = diagnose_residuals(model.resid)
    print(f'\n--- {label} Residuals ---')
    for k, v in diag.items():
        print(f'  {k:25s}: {v:.4f}' if isinstance(v, float) else f'  {k:25s}: {v}')

# ACF / PACF plots (matplotlib)
import matplotlib.pyplot as plt
fig_acf = plot_acf_pacf(arima_fit.resid, lags=40)
fig_acf.suptitle(f'{sym} ARIMA Residuals — ACF & PACF', y=1.02)
plt.show()

fig_acf_s = plot_acf_pacf(sarima_fit.resid, lags=40)
fig_acf_s.suptitle(f'{sym} SARIMA Residuals — ACF & PACF', y=1.02)
plt.show()

In [ ]:
# ═══════════════════════════════════════════════════════════════
# CELL 11: Multicollinearity (VIF)
# ═══════════════════════════════════════════════════════════════
tech_cols = [c for c in df.columns if any(x in c for x in ['SMA','EMA','RSI','MACD','ATR','BB'])]
vif_df = compute_vif(df, tech_cols)
print(vif_df.to_string(index=False))
print('\nNote: VIF > 10 indicates severe multicollinearity.')

In [ ]:
# ═══════════════════════════════════════════════════════════════
# CELL 12: Interactive Visualizations
# ═══════════════════════════════════════════════════════════════
plot_candlestick(df, ticker=sym, show_ma=True, show_bb=True).show()
plot_returns_distribution(df, ticker=sym).show()

In [ ]:
# ═══════════════════════════════════════════════════════════════
# CELL 13: Full One-Call Pipeline
# ═══════════════════════════════════════════════════════════════
results = full_pipeline(
    ticker='TCS.NS',
    exchange='NSE',
    period='2y',
    interval='1d',
    arima_order=(2, 1, 2),
    sarima_seasonal=(1, 1, 1, 252),
    forecast_steps=30,
    verbose=True
)

print('\nKeys:', list(results.keys()))

In [ ]:
# ═══════════════════════════════════════════════════════════════
# CELL 14: BSE Compatibility Demo
# ═══════════════════════════════════════════════════════════════
bse_sym = normalize_ticker('RELIANCE', 'BSE')
print(f'BSE ticker: {bse_sym}')

df_bse = fetch_data(bse_sym, period='6mo', interval='1d')
df_bse = add_all_technicals(df_bse)
print(f'Fetched {len(df_bse)} rows')
df_bse[['Open','High','Low','Close','Volume','Daily_Return']].head(3)

In [ ]:
# ═══════════════════════════════════════════════════════════════
# CELL 15: US Stock Compatibility Demo
# ═══════════════════════════════════════════════════════════════
us_sym = normalize_ticker('AAPL', 'US')
print(f'US ticker: {us_sym}')

df_us = fetch_data(us_sym, period='6mo', interval='1d')
df_us = add_all_technicals(df_us)
print(f'Fetched {len(df_us)} rows')
df_us[['Open','High','Low','Close','Volume','Daily_Return']].head(3)

In [ ]:
# ═══════════════════════════════════════════════════════════════
# CELL 16: Resample & Compare Frequencies
# ═══════════════════════════════════════════════════════════════
df_weekly  = resample_data(df, freq='W')
df_monthly = resample_data(df, freq='ME')

print(f'Daily  : {len(df)} rows')
print(f'Weekly : {len(df_weekly)} rows')
print(f'Monthly: {len(df_monthly)} rows')
df_monthly.head(3)

In [ ]:
# ═══════════════════════════════════════════════════════════════
# CELL 17: ARIMA Grid Search
# ═══════════════════════════════════════════════════════════════
from statsmodels.tsa.arima.model import ARIMA as _ARIMA

best_aic, best_order, best_model = np.inf, None, None

for p in range(0, 4):
    for d in range(0, 2):
        for q in range(0, 4):
            if p == 0 and q == 0:
                continue
            try:
                fitted = _ARIMA(df['Close'], order=(p, d, q)).fit()
                if fitted.aic < best_aic:
                    best_aic, best_order, best_model = fitted.aic, (p, d, q), fitted
            except Exception:
                continue

print(f'Best order: {best_order}  AIC={best_aic:.2f}')
print(best_model.summary().tables[0])

---
### Notebook Complete
All functions in `eda.py` demonstrated. Pipeline is fully NSE / BSE / US compatible.